<a href="https://colab.research.google.com/github/Oaimtac/farm-soccer/blob/main/Python%E7%A8%8B%E5%BC%8F%E8%AA%9E%E8%A8%80_(III)_2_%E8%A9%B3%E8%A7%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Python程式語言 (III)-2**

> 這次的實驗會帶大家使用事前量測好的心電訊號，來進行課程！

> 但不幸地是，量測到的心電訊號被60Hz市電干擾了

> 讓我們來透過上一堂練習的濾波方法來試著觀察並拯救這筆心電訊號吧！


# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
from scipy.fft import fft, fftfreq, ifft #從scipy.fft函式庫中，引入頻域轉換公式fft, fftfreq, ifft
import plotly.graph_objects as go     #引入plotly.graph_objects函式庫，命名為go
import numpy as np             #引入numpy函式庫，命名為np
import pandas as pd            #引入pandas函式庫，命名為np
from google.colab import files      #從google colab函式庫中，引入files函式

# 1.將有雜訊的心電訊號丟進程式庫吧！

In [ ]:
#先將資料夾中的「範例心電訊號(含雜訊)」檔案放到Google雲端處理器中
uploaded = files.upload()            #用uploaded來處理上傳檔案程序
for fn in uploaded.keys():           #將選取所有上傳檔案的名字印出
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

Saving 範例心電訊號(含雜訊).txt to 範例心電訊號(含雜訊) (1).txt
你已經上傳了 "範例心電訊號(含雜訊) (1).txt" 


# 2.看看有雜訊的心電訊號長什麼樣子吧！

In [ ]:
#來讀取剛上傳的資料吧！
df = pd.read_csv('範例心電訊號(含雜訊).txt') #以pandas的read_csv即可讀取檔案
data = np.array(df)              #將讀取到的檔案換成我們習慣的numpy陣列來處理
print(data)                   #觀察data陣列，發現並非是一個一維陣列，而是1*1的二維矩陣

[[91.]
 [89.]
 [92.]
 ...
 [78.]
 [75.]
 [76.]]


In [ ]:
#這樣的二維矩陣不好使用，如何將其轉換成簡易的一維陣列呢？

#讓我們用各個角度來觀察1*1矩陣資料
print("矩陣第一列的資料為：",data[0])
print("矩陣第一列的第一行資料為：",data[0][0])
print("矩陣第二列的資料為：",data[1])
print("矩陣第二列的第一行資料為：",data[1][0])

矩陣第一列的資料為： [91.]
矩陣第一列的第一行資料為： 91.0
矩陣第二列的資料為： [89.]
矩陣第二列的第一行資料為： 89.0


In [ ]:
#觀察中我們發現到，我們只需要把第一行資料取出即可
#numpy函式提供了我們簡便的應用實現這樣的功能！
#透過(,)加數字，即可指定矩陣中的行來建立資料資料

test = np.array([[0,1],[2,3]]) #建立一個測試矩陣
print("test矩陣為：\n",test)
print("test矩陣第一列的資料為：",test[0])
print("test矩陣第二列的資料為：",test[1])
print("test矩陣第一行的資料為：",test[:,0])
print("test矩陣第二行的資料為：",test[:,1])

test矩陣為：
 [[0 1]
 [2 3]]
test矩陣第一列的資料為： [0 1]
test矩陣第二列的資料為： [2 3]
test矩陣第一行的資料為： [0 2]
test矩陣第二行的資料為： [1 3]


In [ ]:
data2 = data[:,0]         #指定data中的第一行資料，建立陣列data2
print("data(1*1)矩陣第一行的資料為：",data2)

data(1*1)矩陣第一行的資料為： [91. 89. 92. ... 78. 75. 76.]


# 3.透過上一個單元學到的頻域轉換公式來觀察雜訊的範圍吧！

In [ ]:
#稍微設定一下畫布資訊！
fig = go.Figure()      #建立一個圖形物件
fig.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data2,        #新增一條線條，將取得的data2畫出
))
fig.update_layout(                  #更新圖形的說明
    title="範例心電訊號(含雜訊)",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                      #顯示圖形

In [ ]:
#上一個單元我們設定了下面這些頻域轉換的參數：
#dots = 3600
#time = np.linspace(0, 1, dots)          #宣告0到1之間的等差數列，最終秒數為1，用於建立不同頻率的sin波
#period = 1/dots                   #設定週期：1秒內有3600個點，頻率就是為3600，週期是頻率的倒數(1/3600)

#在上個單元，我們利用linspace來建立頻率3600的sin波資料；此次心電訊號檔案中，資料的紀錄頻率則是每秒500個點，資料週期是頻率的倒數(1/500)
#dots點數，可直接查看data2陣列的資料長度。

dots = len(data2)                #取得資料總數
period = 1/500                  #宣告週期參數

xf = fftfreq(dots, period)[1: int(dots/2)]   #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
                          #頻域轉換公式只需取前二分之一有效值

yf = fft(data2)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列，
yf_half = np.abs(yf[1: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf_normalized = 2 / dots * yf_half        #頻域轉換後將資料正規畫

fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(x=xf, y=yf_normalized))  #新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
fig.show()                     #顯示圖形

# 4.心電訊號試試看不同的頻域範圍來濾除雜訊吧！



> 透過學術理論的觀察，我們得知心電圖機呈現的心電訊號，通常會取0.5Hz ~ 40Hz的訊號。

> 讓我們把0.5以下，40Hz以上的雜訊，包含市電60Hz的干擾都給濾除吧！



In [ ]:
dots = len(data2)   #取得資料總數
period = 1/500     #宣告週期參數

xf2 = fftfreq(dots, period)            #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
yf2 = fft(data2)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列，

delete_frequency_start1 = 0             #選取特定濾除頻率起點
delete_frequency_end1 =  0.5             #選取特定濾除頻率終點

delete_frequency_start2 = 40             #選取特定濾除頻率起點

for i in range(len(xf2)):      #將選取範圍的頻率能量濾除(設為0)
  if abs(xf2[i])>=delete_frequency_start1 and abs(xf2[i])<=delete_frequency_end1:  #將選取範圍從起點(0)到終點(0.5)的頻率能量濾除
    yf2[i] = 0
    print("已濾除",abs(xf2[i]),"Hz的頻率能量")
  if abs(xf2[i])>=delete_frequency_start2:                      #將選取範圍從起點(40)以上的頻率能量都濾除
    yf2[i] = 0
    print("已濾除",abs(xf2[i]),"Hz的頻率能量")

yf2_half = np.abs(yf2[0: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf2_normalized = 2 / dots * yf2_half        #頻域轉換後將資料正規畫

fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(x=xf2, y=yf2_normalized))  #新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
fig.show()                     #顯示圖形

串流輸出內容已截斷至最後 5000 行。
已濾除 201.64742508756 Hz的頻率能量
已濾除 201.61499545985214 Hz的頻率能量
已濾除 201.58256583214424 Hz的頻率能量
已濾除 201.55013620443637 Hz的頻率能量
已濾除 201.5177065767285 Hz的頻率能量
已濾除 201.48527694902063 Hz的頻率能量
已濾除 201.45284732131276 Hz的頻率能量
已濾除 201.4204176936049 Hz的頻率能量
已濾除 201.38798806589702 Hz的頻率能量
已濾除 201.35555843818915 Hz的頻率能量
已濾除 201.32312881048125 Hz的頻率能量
已濾除 201.29069918277338 Hz的頻率能量
已濾除 201.2582695550655 Hz的頻率能量
已濾除 201.22583992735764 Hz的頻率能量
已濾除 201.19341029964977 Hz的頻率能量
已濾除 201.1609806719419 Hz的頻率能量
已濾除 201.12855104423403 Hz的頻率能量
已濾除 201.09612141652616 Hz的頻率能量
已濾除 201.06369178881826 Hz的頻率能量
已濾除 201.0312621611104 Hz的頻率能量
已濾除 200.99883253340252 Hz的頻率能量
已濾除 200.96640290569465 Hz的頻率能量
已濾除 200.93397327798678 Hz的頻率能量
已濾除 200.9015436502789 Hz的頻率能量
已濾除 200.86911402257104 Hz的頻率能量
已濾除 200.83668439486314 Hz的頻率能量
已濾除 200.80425476715527 Hz的頻率能量
已濾除 200.7718251394474 Hz的頻率能量
已濾除 200.73939551173953 Hz的頻率能量
已濾除 200.70696588403166 Hz的頻率能量
已濾除 200.6745362563238 Hz的頻率能量
已濾除 200.64210662861592 Hz的頻率能

#5.一起來看看濾除60Hz市電雜訊後的心電訊號長什麼樣子吧！

In [ ]:
iyf2 = ifft(yf2, n=np.size(yf2))  #將濾除特定頻率後的訊號，轉換回時域訊號，須提供訊號本身與訊號點數

fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=iyf2.real,                #新增一條線條，，將轉換訊號的時域部分畫出
))
fig.show()                 #顯示圖形

#(3-2 exercise):在學術上，完整的心電訊號PQRST波頻率能量是從0.5到40Hz，但如果只是想要計算心跳速率，只需要有R波就能計算囉！R波的頻率主要集中在15Hz~35Hz。利用本章學習的內容，練習把R波濾出來吧！

In [ ]:
#範例解答
dots = len(data2)   #取得資料總數
period = 1/500     #宣告週期參數

xf3 = fftfreq(dots, period)            #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
yf3 = fft(data2)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列，

delete_frequency_start3 = 0             #選取特定濾除頻率起點
delete_frequency_end3 =  15             #選取特定濾除頻率終點

delete_frequency_start4 = 35             #選取特定濾除頻率起點

for i in range(len(xf3)):      #將選取範圍的頻率能量濾除(設為0)
  if abs(xf3[i])>=delete_frequency_start3 and abs(xf3[i])<=delete_frequency_end3:  #將選取範圍從起點(0)到終點(0.5)的頻率能量濾除
    yf3[i] = 0
    print("已濾除",abs(xf3[i]),"Hz的頻率能量")
  if abs(xf3[i])>=delete_frequency_start4:                      #將選取範圍從起點(40)以上的頻率能量都濾除
    yf3[i] = 0
    print("已濾除",abs(xf3[i]),"Hz的頻率能量")

yf3_half = np.abs(yf3[0: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf3_normalized = 2 / dots * yf3_half        #頻域轉換後將資料正規畫

fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(x=xf3, y=yf3_normalized))  #新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
fig.show()                     #顯示圖形

串流輸出內容已截斷至最後 5000 行。
已濾除 182.1572188351278 Hz的頻率能量
已濾除 182.1247892074199 Hz的頻率能量
已濾除 182.09235957971202 Hz的頻率能量
已濾除 182.05992995200415 Hz的頻率能量
已濾除 182.02750032429628 Hz的頻率能量
已濾除 181.9950706965884 Hz的頻率能量
已濾除 181.96264106888054 Hz的頻率能量
已濾除 181.93021144117267 Hz的頻率能量
已濾除 181.89778181346477 Hz的頻率能量
已濾除 181.8653521857569 Hz的頻率能量
已濾除 181.83292255804903 Hz的頻率能量
已濾除 181.80049293034116 Hz的頻率能量
已濾除 181.7680633026333 Hz的頻率能量
已濾除 181.73563367492542 Hz的頻率能量
已濾除 181.70320404721755 Hz的頻率能量
已濾除 181.67077441950968 Hz的頻率能量
已濾除 181.63834479180179 Hz的頻率能量
已濾除 181.60591516409391 Hz的頻率能量
已濾除 181.57348553638604 Hz的頻率能量
已濾除 181.54105590867817 Hz的頻率能量
已濾除 181.5086262809703 Hz的頻率能量
已濾除 181.47619665326243 Hz的頻率能量
已濾除 181.44376702555456 Hz的頻率能量
已濾除 181.41133739784667 Hz的頻率能量
已濾除 181.3789077701388 Hz的頻率能量
已濾除 181.34647814243093 Hz的頻率能量
已濾除 181.31404851472305 Hz的頻率能量
已濾除 181.28161888701518 Hz的頻率能量
已濾除 181.24918925930731 Hz的頻率能量
已濾除 181.21675963159944 Hz的頻率能量
已濾除 181.18433000389157 Hz的頻率能量
已濾除 181.15190037618368 Hz

In [ ]:
iyf3 = ifft(yf3, n=np.size(yf3))  #將濾除特定頻率後的訊號，轉換回時域訊號，須提供訊號本身與訊號點數

fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=iyf3.real,                #新增一條線條，，將轉換訊號的時域部分畫出
))
fig.show()

# 單元小結論：
---
#1.   提取矩陣的第幾行，可用程式data(:,行數)完成。
#2.   市電雜訊以60Hz為主頻帶，濾除雜訊頻帶，取學術常用心電訊號頻帶0.5~40Hz，可有效濾除市電雜訊。
#3.   將時域訊號透過頻域轉換公式，需要留意資料的總點數與訊號週期。
#4.   心電訊號PRQST的頻率能量為0.5Hz到40Hz；其中R波的能量集中在15Hz到35Hz。